# Sentiment Analysis: Traditional ML vs LLM-based Approaches

**Course:** Napredna analiza podataka

**Student:** Nevena Djordjevic 2022/0052

**Description:**  
This project compares traditional machine learning models and large language models (LLMs) for sentiment analysis on textual data.


## Project Plan

1. Dataset selection and exploration  
2. Data preprocessing and text cleaning  
3. Feature extraction using TF-IDF  
4. Traditional machine learning model for sentiment classification  
5. Zero-shot sentiment classification using a large language model (LLM)  
6. Few-shot sentiment classification using a large language model (LLM)  
7. Model evaluation and comparison  
8. Discussion and conclusions


In [1]:
import sys
print(sys.version)

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


## 1. Dataset selection and exploration

In this project, the IMDb Movie Reviews dataset is used for sentiment classification.
The dataset consists of textual movie reviews collected from the IMDb platform, where
each review is labeled with a binary sentiment value: **positive** or **negative**.

The dataset is a well-known benchmark for text classification tasks and is widely used
for evaluating sentiment analysis models. Each data instance contains:
- a text field representing the movie review
- a label indicating the sentiment polarity

This dataset is suitable for comparing traditional machine learning approaches with
modern large language model (LLM) based methods, such as zero-shot and few-shot
classification.


In [2]:
import zipfile
import os

zip_path = "/content/imdb.zip"
extract_path = "/content/imdb"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
# Extracting ZIP into the target directory

os.listdir(extract_path)
# Quick check: We're expecting a single CSV here

['IMDB Dataset.csv']

In [3]:
import pandas as pd

df = pd.read_csv("/content/imdb/IMDB Dataset.csv")
# Loading the dataset from the extracted CSV file
df.head()
# Preview of the first few rows to verify columns and data format


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df['sentiment'].value_counts()
# Checking if classes are balanced

,count
sentiment,
positive,25000
negative,25000


In [5]:
from sklearn.model_selection import train_test_split

X = df['review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## 2. Data preprocessing and text cleaning

Textual data often contains noise and inconsistencies that can negatively
affect the performance of machine learning models. Therefore, a preprocessing
step is applied to normalize the text and reduce irrelevant variability.

In this project, a minimal and controlled text cleaning procedure is used in
order to preserve as much sentiment-related information as possible. The
following preprocessing steps are applied:
- removal of HTML line break tags (`<br />`),
- conversion of text to lowercase,
- normalization of whitespace.

More aggressive preprocessing steps such as stop-word removal, stemming, or
lemmatization are intentionally not applied at this stage, since they may
remove or distort information relevant for sentiment classification.

This preprocessing strategy ensures a balance between noise reduction and
information preservation, which is particularly important when comparing
traditional machine learning models with large language model (LLM) based
approaches.

Why not stemming or lemmatization?
Because sentiment can be expressed through word forms and intensity, and
aggressive normalization might remove useful signal.

In [6]:
# Checking for missing values in the key columns
df[["review", "sentiment"]].isna().sum()

,0
review,0
sentiment,0


In [7]:
# Quick look at review length distribution (number of characters)
df["review_len"] = df["review"].str.len()

df["review_len"].describe()

,review_len
count,50000.000000
mean,1309.431020
std,989.728014
min,32.000000
25%,699.000000
50%,970.000000
75%,1590.250000
max,13704.000000


In [8]:
import re

def clean_text(text: str) -> str:
    """
    Basic text cleaning for IMDb reviews.
    - Remove HTML line breaks (<br />)
    - Remove extra whitespace
    - Lowercase
    Note: We keep punctuation for now (can help sentiment in some cases).
    """
    text = re.sub(r"<br\s*/?>", " ", text)      # remove HTML breaks
    text = text.lower()                        # lowercase
    text = re.sub(r"\s+", " ", text).strip()   # normalize whitespace
    return text

# Apply cleaning to train/test text
X_train_clean = X_train.apply(clean_text)
X_test_clean = X_test.apply(clean_text)

# Sanity check: show before/after for one example
print("BEFORE:\n", X_train.iloc[0][:300], "\n")
print("AFTER:\n", X_train_clean.iloc[0][:300])


BEFORE:
 I caught this little gem totally by accident back in 1980 or '81. I was at a revival theatre to see two old silly sci-fi movies. The theatre was packed full and (with no warning) they showed a bunch of sci-fi short spoofs (to get us in the mood). Most were somewhat amusing but THIS came on and, with 

AFTER:
 i caught this little gem totally by accident back in 1980 or '81. i was at a revival theatre to see two old silly sci-fi movies. the theatre was packed full and (with no warning) they showed a bunch of sci-fi short spoofs (to get us in the mood). most were somewhat amusing but this came on and, with


## 3. Feature extraction using TF-IDF

Machine learning algorithms require numerical input, therefore textual data
must be transformed into a numeric representation before model training.

In this project, text data is represented using the TF-IDF (Term Frequency –
Inverse Document Frequency) approach. TF-IDF assigns a weight to each word
based on:
- how frequently the word appears in a document (term frequency),
- how rare the word is across the entire corpus (inverse document frequency).

This representation has several advantages for sentiment classification:
- it reduces the impact of very common words (e.g. "the", "movie"),
- it emphasizes words that are more informative for sentiment,
- it is computationally efficient and well-suited for linear classifiers.

TF-IDF is a widely used baseline in text classification tasks and provides a
strong and interpretable foundation for comparison with LLM-based approaches.

Why not word embeddings?
TF-IDF is chosen as a strong and interpretable baseline, which allows a fair comparison with zero-shot and few-shot LLM approaches without introducing additional pretrained semantic knowledge.

### 3.1 TF-IDF vectorization


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,      # limit vocabulary size for efficiency
    ngram_range=(1, 2),      # use unigrams and bigrams
    min_df=5                 # ignore very rare words
)


In [10]:
# Fit TF-IDF on training data only
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_clean)

# Transforming test data using the fitted vectorizer
X_test_tfidf = tfidf_vectorizer.transform(X_test_clean)

# -> No data leakage

In [11]:
X_train_tfidf.shape, X_test_tfidf.shape


((40000, 10000), (10000, 10000))

In [12]:
# Inspect a few feature names
tfidf_vectorizer.get_feature_names_out()[:20]

array(['00', '000', '10', '10 10', '10 for', '10 minutes', '10 out',
       '10 stars', '10 years', '100', '11', '12', '13', '14', '15',
       '15 minutes', '16', '17', '18', '1930'], dtype=object)

## 4. Traditional machine learning model for sentiment classification

In this section, a traditional supervised machine learning approach is applied to the sentiment classification task.
The textual data, previously transformed into numerical representations using the TF-IDF method, is used as input to a classical classification algorithm.

Logistic Regression is selected as the baseline model due to its simplicity, interpretability, and strong performance on high-dimensional sparse text representations such as TF-IDF vectors.
Despite its linear nature, Logistic Regression is widely used as a competitive baseline in sentiment analysis tasks and serves as a strong point of comparison against modern large language model (LLM)-based approaches.

The model is trained on the training subset and evaluated on a held-out test set using standard classification metrics.
This approach provides a clear reference point for assessing the effectiveness of zero-shot and few-shot LLM-based sentiment classification methods in later sections.

In [13]:
# Import machine learning model and evaluation utilities
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


In [14]:
# Initialize Logistic Regression model
log_reg_model = LogisticRegression(
    max_iter=1000,      # increase iterations to ensure convergence
    random_state=42,
    n_jobs=-1           # use all available CPU cores
)


In [15]:
# Train the Logistic Regression model on TF-IDF features
log_reg_model.fit(X_train_tfidf, y_train)


LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42)

In [16]:
# Predict sentiment labels on the test set
y_pred = log_reg_model.predict(X_test_tfidf)


In [17]:
# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label="positive")
recall = recall_score(y_test, y_pred, pos_label="positive")
f1 = f1_score(y_test, y_pred, pos_label="positive")

accuracy, precision, recall, f1

(0.9013, 0.8958374432826988, 0.9082, 0.9019763630946469)

In [18]:
# Detailed classification report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.91      0.89      0.90      5000
    positive       0.90      0.91      0.90      5000

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



**Results and interpretation – Traditional ML model (Logistic Regression)**

In this section, we analyze the performance of the traditional machine learning approach based on TF-IDF feature representation and Logistic Regression for sentiment classification.

The model was trained on 40,000 movie reviews and evaluated on a held-out test set of 10,000 reviews, ensuring a fair and unbiased evaluation.

**Quantitative results**

The obtained evaluation metrics are:

Accuracy: 0.901

Precision (positive class): 0.896

Recall (positive class): 0.908

F1-score (positive class): 0.902

The detailed classification report shows balanced performance across both sentiment classes:

The model achieves similar precision and recall for both positive and negative reviews, indicating no significant bias toward a particular class.

The macro average and weighted average F1-scores are both approximately 0.90, confirming stable performance across classes.

**Interpretation**

These results demonstrate that the traditional TF-IDF + Logistic Regression pipeline is a strong baseline model for sentiment classification:

TF-IDF effectively captures informative word and n-gram patterns relevant to sentiment polarity.

Logistic Regression performs well in high-dimensional sparse feature spaces, making it a suitable choice for text classification tasks.

The achieved F1-score above 0.90 indicates a high level of predictive performance, despite the model’s relative simplicity.

**Importance for further comparison**

The strong performance of this classical machine learning model is particularly important for this study, as it establishes a competitive baseline against which zero-shot and few-shot LLM-based approaches will be compared in the following sections.

Any performance gains achieved by LLM-based methods must therefore be interpreted in light of:

the absence of task-specific training,

increased computational cost,

and reduced interpretability compared to traditional ML models.

## 5. Zero-shot sentiment classification using a Large Language Model (LLM)

Zero-shot classification represents a paradigm in which a model performs a classification task **without being explicitly trained on labeled examples from the target dataset**. Instead of learning from task-specific training data, the model relies on previously acquired linguistic and semantic knowledge obtained during large-scale pretraining.

In the context of sentiment analysis, a zero-shot approach allows a large language model (LLM) to assign sentiment labels (e.g., *positive* or *negative*) to text inputs based solely on a natural language task description and a predefined set of candidate labels.

This approach offers several advantages:
- eliminates the need for labeled training data,
- enables rapid adaptation to new tasks and domains,
- significantly reduces development time.

However, zero-shot classification may also have limitations, such as:
- lower performance compared to supervised models trained on task-specific data,
- sensitivity to prompt formulation and label wording,
- higher computational cost.

In this section, we evaluate the effectiveness of a zero-shot sentiment classification approach using a pretrained large language model and compare its performance with the previously implemented traditional machine learning model.


In [19]:
!pip install -q transformers torch

In [20]:
from transformers import pipeline
import numpy as np

In [21]:
# Initialize zero-shot classification pipeline
zero_shot_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [22]:
# Use a random subset of the test data for a fairer quick evaluation
N_SAMPLES = 200
sample_idx = y_test.sample(n=N_SAMPLES, random_state=42).index

X_test_sample = X_test_clean.loc[sample_idx].tolist()
y_test_sample = y_test.loc[sample_idx].tolist()


In [23]:
# Candidate labels for zero-shot classification
candidate_labels = ["positive", "negative"]

# Optional: hypothesis template (helps the NLI-style model)
hypothesis_template = "This movie review is {}."

zero_shot_preds = []

for text in X_test_sample:
    out = zero_shot_classifier(
        text,
        candidate_labels=candidate_labels,
        hypothesis_template=hypothesis_template,
        multi_label=False
    )
    # out["labels"] is sorted by score descending -> first is predicted label
    zero_shot_preds.append(out["labels"][0])

zero_shot_preds[:10], y_test_sample[:10]


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


(['negative',
  'positive',
  'negative',
  'negative',
  'positive',
  'negative',
  'negative',
  'positive',
  'negative',
  'positive'],
 ['negative',
  'positive',
  'negative',
  'negative',
  'negative',
  'negative',
  'negative',
  'positive',
  'negative',
  'positive'])

In [24]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

zs_acc = accuracy_score(y_test_sample, zero_shot_preds)
zs_prec = precision_score(y_test_sample, zero_shot_preds, pos_label="positive")
zs_rec = recall_score(y_test_sample, zero_shot_preds, pos_label="positive")
zs_f1 = f1_score(y_test_sample, zero_shot_preds, pos_label="positive")

zs_acc, zs_prec, zs_rec, zs_f1


(0.945, 0.9259259259259259, 0.970873786407767, 0.9478672985781991)

In [25]:
print(classification_report(y_test_sample, zero_shot_preds))

              precision    recall  f1-score   support

    negative       0.97      0.92      0.94        97
    positive       0.93      0.97      0.95       103

    accuracy                           0.94       200
   macro avg       0.95      0.94      0.94       200
weighted avg       0.95      0.94      0.94       200



**Zero-shot sentiment classification (Evaluation)**

Zero-shot sentiment classification was evaluated on a randomly selected subset of 200 test samples due to the computational cost of large language models. The facebook/bart-large-mnli model was used without any task-specific fine-tuning.

Despite the absence of supervised training on the IMDb dataset, the model achieved an accuracy of 94.5%, with an F1-score of 0.95 for the positive class. These results indicate that large pre-trained language models are capable of capturing sentiment-related semantic patterns directly from natural language, even in a zero-shot setting.

Important note:
The comparison should be interpreted with caution, as the zero-shot model was evaluated on a smaller subset of the test data due to computational constraints. However, the results still provide valuable insight into the trade-off between performance and computational cost.

## 6. Few-shot sentiment classification using a large language model (LLM)

Few-shot sentiment classification extends the zero-shot approach by providing the model with a small number of labeled examples directly in the prompt. Instead of relying only on general language understanding, the model can use these examples as guidance for how the task should be solved and how the labels should be interpreted in this specific context.

In this project, few-shot classification is implemented by selecting a small, balanced set of IMDb reviews (e.g., 3–5 positive and 3–5 negative) and embedding them into the prompt along with the new, unseen review. The model is then asked to output only one of the predefined labels (positive or negative). This setup allows us to evaluate whether a limited amount of in-context supervision improves performance compared to the zero-shot setting, and how few-shot results compare with the traditional TF-IDF + Logistic Regression baseline.

Because LLM inference is computationally expensive, the few-shot approach is evaluated on the same limited subset of test samples as in the zero-shot experiment, ensuring a fair comparison between the two LLM-based methods.